# GeoSample AI — Exploratory Data Analysis

This notebook walks through the initial data exploration:
- Visualise USGS copper deposit locations on a map
- Preview Sentinel-2 alteration score rasters
- Inspect extracted NI 43-101 text quality


In [ ]:
import sys
sys.path.insert(0, '..')

import geopandas as gpd
import matplotlib.pyplot as plt
import folium
from pathlib import Path

DATA = Path('../data')

## 1. USGS copper deposit locations

In [ ]:
deposits_path = DATA / 'processed/vectors/usgs_copper_deposits.geojson'

if deposits_path.exists():
    gdf = gpd.read_file(deposits_path)
    print(f'Loaded {len(gdf):,} copper deposit records')
    print(gdf.head())
    
    # Quick map
    m = folium.Map(location=[-10, 26], zoom_start=5, tiles='CartoDB positron')
    for _, row in gdf.iterrows():
        if row.geometry:
            folium.CircleMarker(
                location=[row.geometry.y, row.geometry.x],
                radius=4,
                color='#D85A30',
                fill=True,
                fill_opacity=0.7,
                tooltip=str(row.get('NAME', 'Unknown')),
            ).add_to(m)
    display(m)
else:
    print('Run scripts/download_all.py and scripts/run_pipeline.py first')

## 2. Sentinel-2 alteration score raster

In [ ]:
import rioxarray

raster_path = DATA / 'processed/rasters/alteration_score.tif'

if raster_path.exists():
    da = rioxarray.open_rasterio(raster_path).squeeze()
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    da.plot(ax=ax, cmap='YlOrRd', robust=True)
    ax.set_title('Copper Alteration Score (Sentinel-2 Band Ratios)')
    plt.tight_layout()
    plt.show()
else:
    print('Alteration raster not found — run the pipeline first')

## 3. NI 43-101 text quality check

In [ ]:
text_dir = DATA / 'processed/text'
txt_files = list(text_dir.glob('*.txt'))
print(f'Found {len(txt_files)} extracted report texts')

if txt_files:
    sample = txt_files[0]
    text = sample.read_text(encoding='utf-8')
    print(f'\nSample from: {sample.name}')
    print(f'Total characters: {len(text):,}')
    print('\n--- First 1000 chars ---')
    print(text[:1000])